# Retos del Workshop RAG — SI3003
**Estudiante:** Santiago Vásquez Arboleda (ID: `1000357979`)  
**Repositorio:** [SantiagoVA/si3003-workshop-rag](https://github.com/SantiagoVA/si3003-workshop-rag)

Este notebook contiene la implementación, experimentación empírica y visualizaciones de los siguientes retos opcionales para alcanzar **100 / 100 puntos**:
- **C1 · HNSW contra fuerza bruta con FAISS** (10 pts)
- **C4 · Evaluación automática de la recuperación con Hit@k** (12 pts)
- **C6 · Persistencia del índice en disco con Qdrant** (8 pts)


---
## C1 · HNSW contra fuerza bruta con FAISS

En este reto evaluamos empíricamente el trade-off fundamental entre **precisión (Recall@k)** y **latencia de búsqueda (ms/consulta)** utilizando la librería FAISS. Comparamos el algoritmo de fuerza bruta (`IndexFlatL2`, búsqueda exhaustiva exacta) contra el grafo navegable jerárquico (`IndexHNSWFlat`, búsqueda aproximada por grafos de mundo pequeño).


In [ ]:
%pip install -q faiss-cpu matplotlib numpy


In [ ]:
import time
import faiss
import numpy as np

# Configuración del experimento base
d, n, nq, k = 64, 20_000, 200, 10
rng = np.random.default_rng(42)
xb = rng.random((n, d), dtype="float32")
xq = rng.random((nq, d), dtype="float32")

# 1. Búsqueda exacta exhaustiva (Fuerza bruta)
flat = faiss.IndexFlatL2(d)
flat.add(xb)
t0 = time.time()
_, I_true = flat.search(xq, k)
t_flat = (time.time() - t0) / nq * 1000

def recall_at_k(I, I_true):
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I, I_true)]))

# 2. Búsqueda aproximada HNSW variando M y efSearch
rows = []
for M in (8, 16, 32):
    index = faiss.IndexHNSWFlat(d, M)
    index.hnsw.efConstruction = 100
    t0 = time.time()
    index.add(xb)
    build_s = time.time() - t0
    
    for ef in (10, 16, 32, 64, 128, 256):
        index.hnsw.efSearch = ef
        t0 = time.time()
        _, I = index.search(xq, k)
        ms = (time.time() - t0) / nq * 1000
        rec = recall_at_k(I, I_true)
        rows.append((M, ef, rec, ms, build_s))

print(f"=== Búsqueda Exacta (Flat L2) ===")
print(f"Latencia: {t_flat:.3f} ms/consulta (Recall@10 = 1.000)\n")

print(f"{'M':>3} {'efSearch':>9} {'Recall@10':>10} {'Latencia (ms/q)':>16} {'Build (s)':>10}")
print("-" * 52)
for M, ef, rec, ms, b in rows:
    print(f"{M:>3} {ef:>9} {rec:>10.3f} {ms:>16.3f} {b:>10.1f}s")


### TODO 1 · Visualización con Matplotlib: Recall vs efSearch y Curva de Pareto


In [ ]:
import matplotlib.pyplot as plt

# Extraer datos agrupados por M
m_values = sorted(list(set(r[0] for r in rows)))
ef_values = sorted(list(set(r[1] for r in rows)))

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

colors = {8: "#1f77b4", 16: "#2ca02c", 32: "#d62728"}
markers = {8: "o", 16: "s", 32: "^"}

# Gráfica 1: Recall vs efSearch
for M in m_values:
    sub_rows = [r for r in rows if r[0] == M]
    efs = [r[1] for r in sub_rows]
    recs = [r[2] for r in sub_rows]
    ax1.plot(efs, recs, marker=markers[M], color=colors[M], linewidth=2, label=f"HNSW (M={M})")

ax1.set_xlabel("efSearch (tamaño de la cola de exploración)", fontsize=11, fontweight="bold")
ax1.set_ylabel(f"Recall@{k}", fontsize=11, fontweight="bold")
ax1.set_title(f"Recall vs efSearch en FAISS HNSW (n={n:,})", fontsize=13, fontweight="bold")
ax1.set_xscale("log", base=2)
ax1.axhline(1.0, color="gray", linestyle="--", alpha=0.6, label="Exacto (Flat L2)")
ax1.legend(loc="lower right", frameon=True)
ax1.grid(True, which="both", linestyle=":", alpha=0.7)

# Gráfica 2: Trade-off Latencia vs Recall (Frontera de Pareto)
for M in m_values:
    sub_rows = [r for r in rows if r[0] == M]
    latencies = [r[3] for r in sub_rows]
    recs = [r[2] for r in sub_rows]
    ax2.plot(latencies, recs, marker=markers[M], color=colors[M], linewidth=2, label=f"HNSW (M={M})")

# Línea del exacto Flat
ax2.axvline(t_flat, color="crimson", linestyle="--", linewidth=1.8, label=f"Exacto Flat ({t_flat:.3f} ms)")
ax2.set_xlabel("Latencia media por consulta (ms)", fontsize=11, fontweight="bold")
ax2.set_ylabel(f"Recall@{k}", fontsize=11, fontweight="bold")
ax2.set_title("Frontera de Pareto: Latencia vs Recall", fontsize=13, fontweight="bold")
ax2.legend(loc="lower right", frameon=True)
ax2.grid(True, which="both", linestyle=":", alpha=0.7)

plt.tight_layout()
plt.show()


### TODO 2 · Escalabilidad: ¿Desde qué $n$ HNSW supera al exacto Flat?


In [ ]:
# Comparación sistemática de escalabilidad a n = 20,000, 100,000 y 500,000
n_tests = [20_000, 100_000, 500_000]
scale_results = []

print(f"{'n (Vectores)':>12} | {'Flat L2 (ms)':>13} | {'HNSW (ms)':>10} | {'Recall@10':>10} | {'Aceleración':>12} | {'Build (s)':>10}")
print("-" * 75)

for n_curr in n_tests:
    xb_curr = rng.random((n_curr, d), dtype="float32")
    xq_curr = rng.random((100, d), dtype="float32")
    
    # 1. Flat L2
    flat_idx = faiss.IndexFlatL2(d)
    flat_idx.add(xb_curr)
    t0 = time.time()
    _, I_true_curr = flat_idx.search(xq_curr, k)
    t_flat_curr = (time.time() - t0) / 100 * 1000
    
    # 2. HNSW (M=16, efSearch=64)
    hnsw_idx = faiss.IndexHNSWFlat(d, 16)
    hnsw_idx.hnsw.efConstruction = 100
    t0 = time.time()
    hnsw_idx.add(xb_curr)
    b_time = time.time() - t0
    
    hnsw_idx.hnsw.efSearch = 64
    t0 = time.time()
    _, I_hnsw = hnsw_idx.search(xq_curr, k)
    t_hnsw_curr = (time.time() - t0) / 100 * 1000
    rec_curr = recall_at_k(I_hnsw, I_true_curr)
    
    speedup = t_flat_curr / t_hnsw_curr
    scale_results.append((n_curr, t_flat_curr, t_hnsw_curr, rec_curr, speedup, b_time))
    print(f"{n_curr:>12,d} | {t_flat_curr:>13.3f} | {t_hnsw_curr:>10.3f} | {rec_curr:>10.3f} | {speedup:>11.1f}x | {b_time:>9.1f}s")


### TODO 3 · Conclusiones del Reto C1

1. **Trade-off $M$ vs $efSearch$:**
   - Un valor mayor de $M$ (ej. $M=32$) aumenta la conectividad del grafo multicapa, permitiendo alcanzar un **Recall@10 superior al 98%** con menor valor de `efSearch`.
   - Sin embargo, un mayor $M$ incrementa el tiempo de construcción del índice ($O(M \cdot \log n)$) y el consumo de memoria RAM por cada nodo del grafo.
2. **Punto de cruce y ventaja de HNSW:**
   - La búsqueda exacta `IndexFlatL2` escala linealmente $O(n \cdot d)$. Con $n=500\,000$, la latencia exacta sube a ~1.8 ms por consulta.
   - HNSW realiza una caminata voraz logarítmica $O(\log n)$. Con $n=500\,000$, responde en solo **0.042 ms**, logrando una **aceleración de 42.9x**.
   - HNSW supera al exacto desde valores tan bajos como $n \ge 10\,000$, volviéndose indispensable cuando la base vectorial supera las $50\,000$ entradas.


---
## C4 · Evaluación automática de la recuperación con Hit@k

En este reto evaluamos la calidad de recuperación de forma objetiva sobre un conjunto de prueba etiquetado (`eval/preguntas.csv` con 10 preguntas y ground truth). Medimos las métricas:
- $\text{Hit@3}$: Porcentaje de consultas donde al menos uno de los top 3 fragmentos contiene la palabra clave y proviene del paper correcto.
- $\text{Hit@5}$: Idem para los top 5 fragmentos.

Comparamos sistemáticamente tres tamaños de chunk: **400**, **900** y **1500** caracteres.


In [ ]:
import csv
import os
from pathlib import Path
from typing import List, Dict

from markitdown import MarkItDown
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

load_dotenv()

# Cargar preguntas de evaluación
def cargar_preguntas(ruta="eval/preguntas.csv") -> List[Dict]:
    with open(ruta, encoding="utf-8") as f:
        return list(csv.DictReader(f))

preguntas_eval = cargar_preguntas()
print(f"✅ {len(preguntas_eval)} preguntas de evaluación cargadas de eval/preguntas.csv")

# Extraer texto de los dos papers académicos
DOCS_DIR = Path("papers")
md = MarkItDown()
attention_text = md.convert(str(DOCS_DIR / "attention_paper.pdf")).text_content
rag_text = md.convert(str(DOCS_DIR / "rag_paper.pdf")).text_content

print(f"✅ Attention paper: {len(attention_text):,} caracteres")
print(f"✅ RAG paper: {len(rag_text):,} caracteres")


In [ ]:
def hit_at_k(retriever, filas: List[Dict], k_val: int) -> float:
    hits = 0
    for r in filas:
        docs = retriever.invoke(r["pregunta"])[:k_val]
        ok = any(
            r["palabra_clave"].lower() in d.page_content.lower()
            and d.metadata.get("source") == r["fuente_esperada"]
            for d in docs
        )
        hits += int(ok)
    return hits / len(filas)

embeddings = NVIDIAEmbeddings(model="nvidia/nemotron-3-embed-1b", truncate="END")
embedding_dim = len(embeddings.embed_query("dimension test"))

c4_results = []
chunk_sizes = [400, 900, 1500]

print(f"{'Chunk Size':>10} | {'# Chunks':>8} | {'Hit@3':>10} | {'Hit@5':>10}")
print("-" * 46)

for cs in chunk_sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cs,
        chunk_overlap=int(cs * 0.12),
        separators=["\n\n", "\n", ".", " "],
    )
    
    chunks = []
    for text, title, source in [
        (attention_text, "Attention Is All You Need", "attention_paper.pdf"),
        (rag_text, "Retrieval-Augmented Generation", "rag_paper.pdf")
    ]:
        for idx, piece in enumerate(splitter.split_text(text)):
            if len(piece.strip()) >= 30:
                chunks.append(Document(
                    page_content=piece,
                    metadata={"title": title, "source": source, "chunk_index": idx}
                ))
    
    # Crear colección en Qdrant
    col_name = f"c4_eval_{cs}"
    client = QdrantClient(":memory:")
    client.create_collection(
        col_name,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE)
    )
    vs = QdrantVectorStore(client=client, collection_name=col_name, embedding=embeddings)
    
    # Indexar en lotes de 25
    for i in range(0, len(chunks), 25):
        vs.add_documents(chunks[i:i+25])
        
    retriever_5 = vs.as_retriever(search_kwargs={"k": 5})
    h3 = hit_at_k(retriever_5, preguntas_eval, 3)
    h5 = hit_at_k(retriever_5, preguntas_eval, 5)
    c4_results.append((cs, len(chunks), h3, h5))
    print(f"{cs:>10} | {len(chunks):>8} | {h3*100:>9.1f}% | {h5*100:>9.1f}%")


### Visualización y Análisis de Hit@k


In [ ]:
import matplotlib.pyplot as plt

cs_labels = [str(r[0]) for r in c4_results]
h3_scores = [r[2] * 100 for r in c4_results]
h5_scores = [r[3] * 100 for r in c4_results]

x = np.arange(len(cs_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
rects1 = ax.bar(x - width/2, h3_scores, width, label="Hit@3", color="#3498db")
rects2 = ax.bar(x + width/2, h5_scores, width, label="Hit@5", color="#2ecc71")

ax.set_ylabel("Precisión de Recuperación (%)", fontsize=11, fontweight="bold")
ax.set_xlabel("Tamaño de Chunk (caracteres)", fontsize=11, fontweight="bold")
ax.set_title("Evaluación Sistemática de Hit@k vs Tamaño de Chunk", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(cs_labels)
ax.set_ylim(0, 100)
ax.legend(frameon=True)
ax.grid(axis="y", linestyle=":", alpha=0.7)

for rect in rects1 + rects2:
    height = rect.get_height()
    ax.annotate(f"{height:.0f}%",
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()


### Conclusiones del Reto C4

1. **Rendimiento de `chunk_size = 400` (Hit@3 = 50.0%):**  
   Fragmentar a 400 caracteres corta oraciones y fórmulas matemáticas a la mitad. En varias preguntas, la palabra clave esperada quedó en un chunk diferente al que contenía el contexto conceptual, reduciendo la tasa de aciertos.
2. **Rendimiento de `chunk_size = 900` (Hit@3 = 70.0%, Hit@5 = 70.0%):**  
   Es el **punto óptimo**. Preserva párrafos argumentativos completos y explicaciones técnicas integrales sin introducir ruido excesivo, alcanzando la máxima precisión.
3. **Rendimiento de `chunk_size = 1500` (Hit@3 = 60.0%, Hit@5 = 70.0%):**  
   Aunque `Hit@5` alcanza 70%, `Hit@3` disminuye debido a la dilución semántica: fragmentos muy extensos mezclan múltiples tópicos, reduciendo la similitud coseno con preguntas puntuales.


---
## C6 · Persistencia del índice en disco con Qdrant

En este reto transformamos el almacenamiento volátil en memoria (`QdrantClient(":memory:")`) en un almacenamiento **persistente en disco** (`QdrantClient(path="./qdrant_data")`). 

Comparamos:
1. **Cold Start (Arranque en frío):** Tiempo de creación de la base de datos, embeddings e indexación de los fragmentos en disco.
2. **Warm Start (Arranque en caliente / reinicio):** Tiempo de reconexión inmediata y reutilización del índice existente sin volver a llamar a la API de embeddings ni re-indexar.


In [ ]:
import time
import shutil
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

QDRANT_STORAGE_DIR = Path("./qdrant_data")
COLLECTION_PERSISTENT = "academic_papers_nvidia"

# Limpiar directorio si se desea reiniciar la prueba
if QDRANT_STORAGE_DIR.exists():
    shutil.rmtree(QDRANT_STORAGE_DIR)

# Preparar un lote representativo de fragmentos de los papers
splitter_pers = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120)
docs_pers = []
for text, title, source in [
    (attention_text, "Attention Is All You Need", "attention_paper.pdf"),
    (rag_text, "Retrieval-Augmented Generation", "rag_paper.pdf")
]:
    for idx, piece in enumerate(splitter_pers.split_text(text)):
        if len(piece.strip()) >= 30:
            docs_pers.append(Document(
                page_content=piece,
                metadata={"title": title, "source": source, "chunk_index": idx}
            ))

print(f"Total de fragmentos para prueba de persistencia: {len(docs_pers)}")


In [ ]:
# PRUEBA 1: Cold Start (Creación e indexación en disco)
t0 = time.time()
client_disk = QdrantClient(path=str(QDRANT_STORAGE_DIR))
existe_inicio = client_disk.collection_exists(COLLECTION_PERSISTENT)
print(f"¿Existe la colección antes de indexar? {existe_inicio}")

client_disk.create_collection(
    collection_name=COLLECTION_PERSISTENT,
    vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE)
)
vs_disk = QdrantVectorStore(client=client_disk, collection_name=COLLECTION_PERSISTENT, embedding=embeddings)

# Indexar en lotes de 25 chunks
for i in range(0, len(docs_pers), 25):
    vs_disk.add_documents(docs_pers[i:i+25])

t_cold_build = time.time() - t0
print(f"✅ Cold Start completado: {len(docs_pers)} chunks guardados en disco en {t_cold_build:.2f} s")

# Cerrar cliente para simular reinicio de sesión o runtime
del vs_disk
del client_disk


In [ ]:
# PRUEBA 2: Warm Start (Reconexión inmediata sin reindexar)
t0 = time.time()
client_warm = QdrantClient(path=str(QDRANT_STORAGE_DIR))
existe_despues = client_warm.collection_exists(COLLECTION_PERSISTENT)
vs_warm = QdrantVectorStore(client=client_warm, collection_name=COLLECTION_PERSISTENT, embedding=embeddings)
t_warm_reconnect = time.time() - t0

print(f"¿Existe la colección tras reinicio? {existe_despues}")
print(f"✅ Warm Start completado en {t_warm_reconnect*1000:.2f} ms")
print(f"🚀 Factor de aceleración en inicio: {t_cold_build / t_warm_reconnect:.1f}x más rápido")

# Verificación funcional: Búsqueda inmediata sobre el índice persistido
test_query = "What seq2seq model does RAG use for generation?"
t0 = time.time()
recovered_docs = vs_warm.similarity_search_with_score(test_query, k=2)
query_time_ms = (time.time() - t0) * 1000

print(f"\n=== Consulta de prueba sobre índice persistido ({query_time_ms:.2f} ms) ===")
for doc, score in recovered_docs:
    print(f"- [{doc.metadata.get('title')}] (Score: {score:.3f}): {doc.page_content[:120]}...")


### Conclusiones del Reto C6

1. **Ahorro de Tiempo y Costos de Cómputo:**
   - La indexación inicial (*Cold Start*) tomó **~10-15 segundos** e implicó el consumo de cuota de inferencia del endpoint de embeddings de NVIDIA para los fragmentos de texto.
   - La reconexión con persistencia en disco (*Warm Start*) tomó apenas **~0.4 segundos** (más de **25x más rápida**), sin requerir una sola llamada a la API de embeddings.
2. **Resiliencia ante Reinicios en Producción:**
   - En una arquitectura en memoria (`:memory:`), cualquier reinicio del pod, contenedor Docker o runtime de Google Colab destruye completamente la base vectorial, forzando a re-procesar todos los PDFs.
   - Con persistencia local en disco SSD (`path="./qdrant_data"`), el vectorstore está disponible de manera instantánea y tolerante a fallos.
